# Notebook 02: Scaled Dot-Product Attention

**Module:** 10 Transformers  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Derive scaled dot-product attention
2. Implement from scratch in NumPy
3. Explain why scale by sqrt(d_k)
4. Apply causal mask for decoders

## Scaled Dot-Product Attention (Vaswani et al., 2017)

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

**Shapes:** $Q, K, V \in \mathbb{R}^{\text{seq} \times d_k}$

**Why scale?** Dot products grow with $d_k$ → softmax saturates → tiny gradients. Dividing by $\sqrt{d_k}$ keeps variance stable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """NumPy: Q,K,V shapes (..., seq, d_k). Returns output, weights."""
    d_k = Q.shape[-1]
    scores = np.matmul(Q, np.swapaxes(K, -2, -1)) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = weights / weights.sum(axis=-1, keepdims=True)
    out = np.matmul(weights, V)
    return out, weights

In [ ]:
seq, d_k, d_v = 4, 8, 8
Q = np.random.randn(seq, d_k)
K = np.random.randn(seq, d_k)
V = np.random.randn(seq, d_v)
out, w = scaled_dot_product_attention(Q, K, V)
print('Output shape:', out.shape)
print('Weights sum per row:', w.sum(axis=1))

plt.imshow(w, cmap='Blues'); plt.colorbar()
plt.xticks(range(seq), ['t0','t1','t2','t3']); plt.yticks(range(seq), ['t0','t1','t2','t3'])
plt.title('Attention weights'); plt.show()

## Causal Mask (Decoder)

Autoregressive models (GPT) mask future tokens:

```
[[1, 0, 0, 0],
 [1, 1, 0, 0],
 [1, 1, 1, 0],
 [1, 1, 1, 1]]
```

In [ ]:
seq = 4
causal = np.tril(np.ones((seq, seq))).astype(bool)
out_causal, w_causal = scaled_dot_product_attention(Q, K, V, mask=causal)
plt.imshow(w_causal, cmap='Blues'); plt.title('Causal attention'); plt.show()

---

## Summary

Scaled dot-product attention is the core operation of all transformers.

**Next:** [03_Self_Attention.ipynb](03_Self_Attention.ipynb)